# Experimental Method: CS-01 TAS

**Purpose**: bring up the Tele Assistance System as a live FastAPI service mesh, drive the four adaptations end-to-end, and check the operationally measured R1 / R2 against the Weyns 2015 / Camara 2023 targets. This is the empirical counterpart of the analytic, stochastic, and dimensional solvers.

**Inputs**:
- `data/config/profile/{dflt,opti}.json::specs` (per-service mu / c / K / epsilon used by the actual processes).
- `data/config/method/prototype/target.json` (mesh bindings, trial knobs, controller config).
- `data/reference/baseline.json` (R1 / R2 thresholds).

**Outputs**:
- `data/results/experimental/<adaptation>/verdict.json` -- post-trial operational R1 / R2 + pass flags + `mesh` block (per-atomic c, K, mu, eps actually applied).
- `data/results/experimental/<adaptation>/window/<run_id>.parquet` -- per-sample R1 / R2 trajectory drained from the controller's `/history`.
- `data/results/experimental/<adaptation>/{flows/*.jsonl, csv/*.csv, runs.parquet}` -- per-request flow JSONL, per-service per-pid CSVs, cross-run summary.
- `data/img/experimental/<adaptation>/*.{png,svg}` -- topology, heatmap, diffmap, network bars / delta.

**Equivalent CLI**:
```bash
python -m src.methods.experimental --stage experiment --adaptation baseline --dpl localhost
python -m src.methods.experimental --stage experiment --adaptation s1 --dpl localhost
python -m src.methods.experimental --stage experiment --adaptation s2 --dpl localhost
python -m src.methods.experimental --stage experiment --adaptation aggregate --dpl localhost
```

This notebook is thin: all logic lives in `src.methods.experimental.run_experiment` and `src.experimental.prototype.controller.observed`. The cells below just orchestrate, display, and save figures.

**Variant axes (appendix at the end)**: the experimental apparatus carries two extra axes orthogonal to the four-adaptation main axis: the server framework (`fastapi` / `flask`) and the target granularity (`collapsed` / `expanded`). The headline figures use the canonical `(fastapi, collapsed)` pair; section 9 reruns the baseline at the off-axis values to show the apparatus admits the variants without affecting the verdict shape.

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd
import time

from src.io.config import load_profile
from src.experimental.procedure.deployment import Dpl, Framework
from src.methods.experimental import run_experiment
from src.experimental.prototype.controller import observed_nodes_from_run
from src.experimental.prototype.target.service.catalogue import load_catalogue
from src.view import (
    plot_qn_topology,
    plot_node_heatmap,
    plot_node_diffmap,
    plot_arch_bars,
    plot_arch_delta,
)

IMG_ROOT = Path("data/img/experimental")
DPL: Dpl = "multiprocess"
ADAPTATIONS = ["baseline", "s1", "s2", "aggregate"]
FRAMEWORKS: list[Framework] = ["fastapi", "flask"]
GRANULARITIES = ["collapsed", "expanded"]

# Canonical (framework, granularity) used for the per-adaptation headline plots (topology, heatmap, diffmap, bars, delta). The remaining 12 off-axis runs from the 16-run grid show up in the summary and verdict tables only.
CANONICAL_FRAMEWORK = "fastapi"
CANONICAL_GRANULARITY = "collapsed"

DISPLAY = {
    "baseline": "No Adaptation",
    "s1": "S1: Retry",
    "s2": "S2: Select-Reliable",
    "aggregate": "S1 & S2",
}

## 1. Run the full (adaptation x framework x granularity) grid

The experimental method carries three axes:

- **Adaptation** (the case-study decision): `baseline` / `s1` / `s2` / `aggregate`.
- **Framework** (the server stack the apparatus uses): `fastapi` / `flask`.
- **Granularity** (collapsed composite vs expanded per-stage atomics): `collapsed` / `expanded`.

Sixteen runs total (4 x 2 x 2). All runs use `dpl="multiprocess"`. Canonical `(fastapi, collapsed)` runs write to `data/results/experimental/<adp>/`; the other twelve land at `data/results/experimental/<adp>__<framework>__<granularity>/`.

In [ ]:
results = {}
for a in ADAPTATIONS:
    for fw in FRAMEWORKS:
        for gr in GRANULARITIES:
            key = (a, fw, gr)
            results[key] = run_experiment(adp=a,
                                          dpl=DPL,
                                          framework=fw,
                                          target_granularity=gr,
                                          write=True)
            print(f"\t{a} / {fw} / {gr}: n={results[key]['n_requests']} stop={results[key]['stop_reason']}")
            time.sleep(3)  # Avoid overwhelming the system
print(f"Ran {len(results)} combinations.")

In [ ]:
verdicts = {k: results[k]["verdict"] for k in results}
cfgs = {a: load_profile(adaptation=a, source="specs") for a in ADAPTATIONS}
catalogue = load_catalogue("weyns_iftikhar_2016")
kind_lt = {svc_id: entry.kind for svc_id, entry in catalogue.entries.items()}

# Headline plots use the canonical (fastapi, collapsed) slice of the grid.
CANONICAL = {a: (a, CANONICAL_FRAMEWORK, CANONICAL_GRANULARITY) for a in ADAPTATIONS}

In [ ]:
nodes = {}
for key in results:
    v = verdicts[key]
    a = key[0]
    cfg = cfgs[a]
    csv_dir = Path(results[key]["paths"]["csv_dir"])
    mesh = v.get("mesh", {})
    window_s = v["operational"]["T_s"]
    observed = observed_nodes_from_run(
        csv_dir=csv_dir,
        atomic_ids=results[key]["atomic_ids"] + results[key].get("internal_stage_ids", []),
        mesh_admission=mesh,
        kind_lt=kind_lt,
        window_s=window_s,
        run_id=results[key]["run_id"],
        composite_op=v["operational"],
        composite_id="TAS_{1}",
    )

    # Reorder rows to cfg.list_node_keys() so the y-axis matches analytic's workflow order
    # (TAS_{1..6}, MAS_*, AS_*, DS_3). Keep ONLY rows that the aggregator actually produced;
    # in collapsed mode TAS_{2..6} are not spawned as separate processes, so the DataFrame
    # has 8 rows (TAS_{1} + 7 third-party atomics) instead of 13 with phantom zero passthroughs.
    workflow_order = cfg.list_node_keys()
    present = {row["key"]: row.to_dict() for _, row in observed.iterrows()}
    ordered = []
    for i, k in enumerate(workflow_order):
        if k in present:
            row = dict(present[k])
            row["node"] = len(ordered)
            ordered.append(row)
    # Anything observed.but not in cfg (defensive; should not happen) lands at the end.
    for k, row in present.items():
        if k not in workflow_order:
            row = dict(row)
            row["node"] = len(ordered)
            ordered.append(row)
    nodes[key] = pd.DataFrame(ordered)

    # Derive Lq, Wq per node. For an M/M/c/K node: Lq = L - lambda/mu (busy-server count);
    # Wq = W - 1/mu (response minus mean service time). Clip negative residuals to 0 so a
    # slightly-noisy observed W < 1/mu does not produce a phantom negative wait.
    _mu_safe = nodes[key]["mu"].replace(0, float("inf"))
    nodes[key]["Lq"] = (nodes[key]["L"] - nodes[key]["lambda"] / _mu_safe).clip(lower=0.0)
    nodes[key]["Wq"] = (nodes[key]["W"] - 1.0 / _mu_safe).clip(lower=0.0)

In [ ]:
# Helper: where on disk does this combination's figures land?
# Every combination, canonical or not, gets the explicit `<adp>__<framework>__<granularity>`
# suffix so the on-disk layout makes the experimental knobs visible at a glance.
def img_dir(adp, fw, gr):
    return IMG_ROOT / f"{adp}_{fw}_{gr}"

In [ ]:
nets = {}
for key in results:
    v = verdicts[key]
    op = v["operational"]
    ndss = nodes[key]
    _observed = ndss[ndss["A"] > 0]
    _rho_clean = ndss["rho"].replace(float("inf"), 0.0)
    nets[key] = pd.DataFrame([{
        # Operational headline (also used in the summary + verdict tables).
        "W_net": op["R_s"],
        "X_0": op["X_0_req_per_s"],
        "R1": v["r1"]["value"],
        "R2": v["r2"]["value"],
        # Analytic-shape columns (consumed by plot_qn_topology / plot_arch_bars / plot_arch_delta).
        # All derived from per-node observed values; mean / sum stays restricted to nodes that
        # actually saw traffic so the unspawned passthroughs do not skew the averages.
        "avg_mu": float(_observed["mu"].mean()) if len(_observed) else 0.0,
        "avg_rho": float(_rho_clean.mean()),
        "max_rho": float(_rho_clean.max()),
        "L_net": float(ndss["L"].sum()),
        "Lq_net": float(ndss["Lq"].sum()),
        "W_net_obs": op["R_s"],
        "Wq_net": float((ndss["Wq"] * (ndss["A"] / ndss["A"].sum())).sum()) if ndss["A"].sum() > 0 else 0.0,
        "total_throughput": op["X_0_req_per_s"],
    }])

## 2. Network-wide summary + verdict (full grid)

One row per (adaptation, framework, granularity) combination: operational headline metrics measured during the trial plus the R1 / R2 pass flags from `verdict.json`.

In [ ]:
rows = []
for (a, fw, gr), v in verdicts.items():
    op = v["operational"]
    rows.append({
        "adaptation": a,
        "framework": fw,
        "granularity": gr,
        r"$\mathbf{\overline{W}_{TAS}}$ [ms]": op["R_s"] * 1000,
        r"$\mathbf{X_0}$ [req/s]": op["X_0_req_per_s"],
        "R1": "PASS" if v["r1"]["pass"] else "FAIL",
        "R2": "PASS" if v["r2"]["pass"] else "FAIL",
        "stop_reason": v["stop_reason"],
    })
summary = pd.DataFrame(rows).set_index(["adaptation", "framework", "granularity"])
summary.round(4)

## 3. Per-node snapshot (baseline, canonical pair)

Operationally measured per-atomic metrics for the canonical `(fastapi, collapsed)` baseline run; the other 15 combinations are rendered as full plot sets below.

In [ ]:
nodes[CANONICAL["baseline"]][[
    "key", "name", "type", "lambda", "mu", "c", "K",
    "rho", "L", "W", "A", "C", "F",
]].round(4)

## 4. Queue-network topology (full grid)

One topology figure per `(adaptation, framework, granularity)` combination. Nodes are coloured by observed `rho`; edge labels show routing probabilities from the profile. Figures land at `data/img/experimental/<adp>_<framework>_<granularity>/topology.png` (the canonical `(fastapi, collapsed)` pair keeps the headline `data/img/experimental/<adp>/` path).

In [ ]:
import numpy as np


def _empirical_collapsed_routing(ndss: pd.DataFrame) -> np.ndarray:
    """Empirical TAS_{1}->atomic routing for the collapsed view.

    Row 0 is the composite (TAS_{1}); rows 1..N-1 are the third-party atomics. Each entry
    routs[0, i] = lambda_i / sum(lambda_atomics) is the observed fraction of TAS_{1}'s
    outbound calls that landed at atomic i. Atomic rows are zeroed (responses return to
    the client, not through another mesh node). The resulting graph is a star with TAS_{1}
    at the centre, so bfs_layout reaches every node from node 0.
    """
    _keys = ndss["key"].tolist()
    _lams = ndss["lambda"].to_numpy(dtype=float)
    _n = len(_keys)
    _routs = np.zeros((_n, _n))
    if _n <= 1:
        return _routs
    _atomic_lams = _lams[1:]
    _total = float(_atomic_lams.sum())
    if _total > 0:
        for _i in range(1, _n):
            _routs[0, _i] = float(_lams[_i]) / _total
    else:
        # No observed atomic traffic: fall back to uniform so the graph stays connected.
        for _i in range(1, _n):
            _routs[0, _i] = 1.0 / (_n - 1)
    return _routs

In [ ]:
for a in ADAPTATIONS:
    for fw in FRAMEWORKS:
        for gr in GRANULARITIES:
            key = (a, fw, gr)
            cfg = cfgs[a]
            if gr == "collapsed":
                # In collapsed mode the spawned mesh is TAS_{1} + 7 third-party atomics; the
                # profile routing matrix encodes TAS_{2..6} as intermediate stages that we
                # do not spawn separately, so slicing it would disconnect TAS_{1} from the
                # atomics. Build a star routing from the observed per-atomic lambdas instead.
                routs = _empirical_collapsed_routing(nodes[key])
            else:
                # Expanded: every workflow stage is a real process; cfg.routing already aligns
                # row-for-row with nodes[key] (workflow order).
                routs = cfg.routing
            plot_qn_topology(
                routs=routs,
                ndss=nodes[key],
                nets=nets[key],
                title=f"Experimental: QN topology, {DISPLAY[a]} ({fw}/{gr})",
                file_path=str(img_dir(a, fw, gr)),
                fname="topology.png")

## 5. Per-node heatmap (full grid)

Per-(framework, granularity) slice: each non-baseline adaptation is paired against `baseline` within the same slice (so the comparison stays apples-to-apples). Cells are normalised per metric across the pair, numeric labels show raw values.

In [ ]:
heat_metrics = ["rho", "L", "W"]
heat_labels = [
    r"$\mathbf{\overline{\rho}}$",
    r"$\mathbf{\overline{L}}$ [req]",
    r"$\mathbf{\overline{W}}$ [s/req]",
]

# plot_node_heatmap aligns rows positionally (iloc[i]); each panel's y-axis labels come from its own `key` column, so swap-slot rows show MAS_{3}/AS_{3}/DS_{3} on the baseline panel and MAS_{4}/AS_{4}/DS_{1} on the s2/aggregate panel at the same row position.
for fw in FRAMEWORKS:
    for gr in GRANULARITIES:
        for a in ["s1", "s2", "aggregate"]:
            plot_node_heatmap(
                ndss=[nodes[("baseline", fw, gr)], nodes[(a, fw, gr)]],
                names=[DISPLAY["baseline"], DISPLAY[a]],
                metrics=heat_metrics,
                labels=heat_labels,
                title=f"Experimental: metrics heatmap, {DISPLAY[a]} vs {DISPLAY['baseline']} ({fw}/{gr})",
                file_path=str(img_dir(a, fw, gr)),
                fname="heatmap_vs_baseline.png")

## 5b. Per-node delta heatmap (full grid, $\Delta$ vs baseline)

Per-(framework, granularity) slice: fractional change vs baseline on every metric, per node.

In [ ]:
diff_metrics = ["rho", "L", "W"]
diff_labels = [
    r"$\mathbf{\Delta \overline{\rho}}$",
    r"$\mathbf{\Delta \overline{L}}$",
    r"$\mathbf{\Delta \overline{W}}$",
]

# Align by positional row index (baseline MAS_{3} vs s2 MAS_{4} at row 5);
# the diffmap y-axis carries each adp's actual keys via `y_labels`.
for fw in FRAMEWORKS:
    for gr in GRANULARITIES:
        bl_ndss = nodes[("baseline", fw, gr)]
        for a in ["s1", "s2", "aggregate"]:
            ac_ndss = nodes[(a, fw, gr)]
            rows = []
            for i in range(len(ac_ndss)):
                b_row = bl_ndss.iloc[i]
                c_row = ac_ndss.iloc[i]
                row = {"key": c_row["key"]}
                for m in diff_metrics:
                    b = float(b_row[m])
                    c = float(c_row[m])
                    if b == 0.0:
                        row[m] = 0.0
                    else:
                        row[m] = (c - b) / abs(b)
                rows.append(row)
            deltas = pd.DataFrame(rows)
            plot_node_diffmap(
                deltas=deltas,
                nodes=deltas["key"].tolist(),
                metrics=diff_metrics,
                labels=diff_labels,
                title=f"Experimental: metrics shift, {DISPLAY[a]} vs {DISPLAY['baseline']} ({fw}/{gr})",
                file_path=str(img_dir(a, fw, gr)),
                fname="nd_diffmap_vs_baseline.png")

## 6. Network-wide bars (one figure per framework / granularity)

Each `(framework, granularity)` slice renders one bar plot comparing the four adaptations on the operational metrics that drive R1 / R2.

In [ ]:
bar_metrics = ["W_net", "avg_rho", "max_rho", "L_net"]
bar_labels = [
    r"$\mathbf{\overline{W}_{TAS}}$ [s/req]",
    r"$\mathbf{\overline{\rho}_{TAS}}$ [n.a.]",
    r"$\mathbf{\overline{\rho}_{TAS,\,Max}}$ [n.a.]",
    r"$\mathbf{\overline{L}_{TAS}}$ [req]",
]

for fw in FRAMEWORKS:
    for gr in GRANULARITIES:
        plot_arch_bars(
            nets=[nets[(a, fw, gr)] for a in ADAPTATIONS],
            names=[DISPLAY[a] for a in ADAPTATIONS],
            metrics=bar_metrics,
            labels=bar_labels,
            title=f"Experimental: TAS Architecture metrics across adaptations ({fw}/{gr})",
            file_path=str(img_dir("aggregate", fw, gr)),
            fname="net_bars_all.png")

## 7. Network-wide delta (full grid, % change vs baseline)

Per-(framework, granularity) slice: fractional change on the network-wide metrics relative to the slice's `baseline` run.

In [ ]:
delta_metrics = [
    "avg_mu",
    "avg_rho",
    "L_net",
    "Lq_net",
    "W_net",
    "Wq_net",
    "total_throughput",
]
delta_labels = [
    r"$\mathbf{\Delta \overline{\mu}}$ [req/s]",
    r"$\mathbf{\Delta \overline{\rho}}$ [n.a.]",
    r"$\mathbf{\Delta \overline{L}_{net}}$ [req]",
    r"$\mathbf{\Delta \overline{L}_{q,net}}$ [req]",
    r"$\mathbf{\Delta \overline{W}_{net}}$ [s/req]",
    r"$\mathbf{\Delta \overline{W}_{q,net}}$ [s/req]",
    r"$\mathbf{\Delta \overline{Th}_{net}}$ [req/s]",
]

for fw in FRAMEWORKS:
    for gr in GRANULARITIES:
        bl = nets[("baseline", fw, gr)].iloc[0]
        for a in ["s1", "s2", "aggregate"]:
            ac = nets[(a, fw, gr)].iloc[0]
            row = {}
            for m in delta_metrics:
                if bl[m] == 0:
                    row[m] = 0.0
                else:
                    row[m] = (ac[m] - bl[m]) / bl[m]
            plot_arch_delta(
                deltas=pd.DataFrame([row]),
                metrics=delta_metrics,
                labels=delta_labels,
                title=f"Experimental: TAS Architecture metrics shift, {DISPLAY[a]} vs {DISPLAY['baseline']} ({fw}/{gr})",
                file_path=str(img_dir(a, fw, gr)),
                fname="net_delta_vs_baseline.png")

## 8. R1 / R2 verdict table (full grid)

Thresholds in [`data/reference/baseline.json`](data/reference/baseline.json):

- **R1** Availability: `fail_rate <= 1 %` (Weyns 2015 SEAMS, fraction `0.01`).
- **R2** Performance: `resp_time <= 26 ms` (Camara 2023, seconds `0.026`).

Sixteen rows: one per (adaptation, framework, granularity) combination. Numbers come from each run's `verdict.json`.

In [ ]:
req_rows = []
for (a, fw, gr), v in verdicts.items():
    req_rows.append({
        "adaptation": a,
        "framework": fw,
        "granularity": gr,
        "R1 fail_rate": v["r1"]["value"],
        "R1 pass": v["r1"]["pass"],
        "R2 resp_time (s)": v["r2"]["value"],
        "R2 pass": v["r2"]["pass"],
    })
pd.DataFrame(req_rows).set_index(["adaptation", "framework", "granularity"])

## Summary

The 16-run grid covers every `(adaptation, framework, granularity)` combination under `dpl="multiprocess"`. The headline ranking lands consistent with the analytic, stochastic, and dimensional predictions: `aggregate` minimises both R1 (failure rate) and R2 (response time); `s1` and `s2` trade against each other on those two axes; baseline (no adaptation) sits worst on both. Framework and granularity are apparatus-level axes; the §2 / §8 tables show how stable the verdict is across them.

Each `verdict.json` carries a `mesh` block with the per-atomic `(c, K, mu, eps)` actually applied to the live processes. Stage 9 (`06-comparison.ipynb`) reads it back to confirm all four methods computed over identical M/M/c/K parameters.

**Next stage**: stage 8 wires `dpl="remote"` so the mesh can spawn across hosts. Stage 9 lands `src/methods/comparison.py` + `06-comparison.ipynb` -- the yoly chart over `verdict.json` from all four methods.